[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C17/E17-1_Plaque_Trouee_FEA.ipynb)

# Plaque trouée : remplacer un calcul par éléments finis

## Le contexte

Une plaque d'acier de 400 mm sur 200 mm et de 20 mm d'épaisseur porte un trou central. Pour dimensionner la pièce, le bureau d'études lance un calcul par éléments finis à chaque diamètre de trou envisagé : il donne la contrainte maximale et le déplacement de la plaque sous une traction de 1 000 N. Un calcul dure en moyenne 9 s, ce qui ne pèse rien pour un diamètre et beaucoup pour un balayage de milliers de variantes. Les 100 calculs disponibles forment un plan de calcul : le diamètre du trou prend cent valeurs régulièrement réparties de 10 à 110 mm. Un substitut, c'est un modèle appris sur ces calculs, qui répond ensuite en une fraction de seconde.

La grandeur à prédire est le déplacement en mm : c'est un problème de régression.

Défi : prédire `Disp/mm` à partir du diamètre du trou et obtenir un R² supérieur à celui du modèle de départ sur la base de test. Ce modèle de départ est une droite ; il atteint un R² de 0,911. À vous de lui faire suivre la courbure. La règle de découpage est fixée plus bas. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Plaque_Trouee_FEA.xlsx`, dans le dossier `donnees` du dépôt : 100 calculs, 12 colonnes. Une ligne est un calcul par éléments finis de la plaque, pour un diamètre de trou donné.

| Colonne | Signification | Unité |
|---|---|---|
| `Unnamed: 0` | numéro du calcul, hérité de l'export, sans signification physique | sans unité |
| `L`, `h`, `k` | longueur, largeur et épaisseur de la plaque, identiques dans tous les calculs | mm |
| `x0`, `y0` | position du centre du trou, identique dans tous les calculs | mm |
| `d0` | diamètre du trou : la seule entrée qui varie | mm |
| `Material` | matériau, identique dans tous les calculs | catégorie |
| `Stress/MPa` | contrainte maximale, seconde sortie du calcul, écartée ici | MPa |
| `Disp/mm` | cible : déplacement maximal de la plaque | mm |
| `Msg` | message du solveur, colonne vide | sans unité |
| `FEA_Runtime` | durée du calcul, une conséquence du calcul et non une entrée | s |

Origine : jeu construit par calculs par éléments finis, sorties du solveur conservées telles quelles. Il enseigne une méthode de substitution, et aucune conclusion de dimensionnement n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on repère les colonnes constantes, la colonne vide et la plage de la cible : la forme d'un fichier se lit sur le fichier entier. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Plaque_Trouee_FEA.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Plaque_Trouee_FEA.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cible = "Disp/mm"
print("dimensions :", df.shape)
constantes = [col for col in df.columns if df[col].nunique(dropna=False) == 1]
print("colonnes constantes ou vides :", constantes)
print("cases vides :", df.isna().sum()[lambda s: s > 0].to_dict())
print("diamètre du trou :", df["d0"].min().round(1), "à", df["d0"].max().round(1), "mm,", df["d0"].nunique(), "valeurs")
print("déplacement :", df[cible].min().round(6), "à", df[cible].max().round(6), "mm")
q1, q3 = df["Stress/MPa"].quantile([0.25, 0.75])
hors_bornes = df["Stress/MPa"] > q3 + 1.5 * (q3 - q1)
print("contrainte hors des bornes de Tukey :", int(hors_bornes.sum()), "calcul sur", len(df))
df.head(3)

Lecture du résultat : 100 calculs, 12 colonnes, mais une seule entrée utile : 7 colonnes sont constantes ou vides (la plaque, la position du trou, le matériau et le message du solveur ne changent pas). Le diamètre prend 100 valeurs régulières, ce qui est un plan de calcul à une variable. Le déplacement va de 500 à 779 nm. Dans la colonne de contrainte, 1 calcul sort des bornes de Tukey : sa valeur, 2,24 MPa, vaut le double de celles des diamètres voisins ; le solveur y a produit un résultat aberrant. Ce calcul figure dans la base de test du contrat, et un seul point de ce genre suffit à plafonner le R² de n'importe quel modèle : le défi porte donc sur le déplacement, que le solveur a calculé sans accident.

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas le dossier `donnees/` du dépôt. La cellule suivante ne télécharge rien : elle cherche ce dossier (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-IA_Ed2026`), vérifie la présence du classeur du défi et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez le dossier `donnees` à la racine de l'espace de travail et téléversez-y le classeur listé, pris dans le dépôt.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
from pathlib import Path

JEUX_LUS = ["MecaIA-Plaque_Trouee_FEA.xlsx"]                  # le classeur lu par ce notebook


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-IA_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees = chercher("donnees")
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Étape 2 : mettre le test de côté

Un quart des calculs est mis de côté avant tout apprentissage, tout réglage et toute comparaison avec la cible. La graine 42 fixe le tirage : chacun obtient la même base de test, et les scores se comparent. Seul le diamètre sert d'entrée : la contrainte est l'autre sortie du calcul et la durée est une conséquence du calcul, aucune des deux n'existe avant lui.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[["d0"]]
y = df[cible]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=GRAINE)
print("apprentissage :", len(X_train), "calculs")
print("test :", len(X_test), "calculs")

### Étape 3 : regarder le déplacement contre le diamètre, sur l'apprentissage

La figure place les calculs d'apprentissage. On compare aussi la pente de la courbe aux petits diamètres et aux grands.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.scatter(X_train["d0"], 1e6 * y_train, s=22, marker="o", color="0.35", edgecolor="black", linewidth=0.4)
ax.set(xlabel="diamètre du trou (mm)", ylabel="déplacement maximal (nm)")
ax.grid(alpha=0.3)
plt.show()
petits, grands = X_train["d0"] < 40, X_train["d0"] > 80
pente_petits = np.polyfit(X_train.loc[petits, "d0"], y_train[petits], 1)[0]
pente_grands = np.polyfit(X_train.loc[grands, "d0"], y_train[grands], 1)[0]
print("pente avant 40 mm :", round(1e6 * pente_petits, 2), "nm par mm ; après 80 mm :", round(1e6 * pente_grands, 2), "nm par mm")

Lecture du résultat : le déplacement croît avec le diamètre, de plus en plus vite : la pente après 80 mm est 7 fois celle d'avant 40 mm. Un trou plus large laisse moins de matière à la plaque, qui se déforme davantage, et cet effet s'accélère quand le ligament restant s'amincit. Une droite ne peut pas suivre cette courbure.

### Étape 4 : construire le modèle de départ

Le modèle de départ est une régression linéaire sur le diamètre : une droite, ajustée sur `X_train` seul.

In [ ]:
# Votre code :

### Étape 5 : mesurer sur la base de test

Le modèle prédit des calculs qu'il n'a jamais vus. Trois indicateurs sont lus : le coefficient de détermination R², et deux erreurs en nm, la racine de l'erreur quadratique moyenne (RMSE) et l'erreur absolue moyenne (MAE). Un témoin sans apprentissage, qui répond toujours la moyenne de l'apprentissage, donne le point de comparaison le plus bas.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

y_pred = modele.predict(X_test)
score_depart = r2_score(y_test, y_pred)
score_temoin = r2_score(y_test, np.full(len(y_test), y_train.mean()))
print(f"R² : {score_depart:.3f}")
print(f"RMSE : {1e6 * root_mean_squared_error(y_test, y_pred):.1f} nm")
print(f"MAE : {1e6 * mean_absolute_error(y_test, y_pred):.1f} nm")
print(f"R² du témoin (toujours la moyenne de l'apprentissage) : {score_temoin:.3f}")

Lecture du résultat : le R² est de 0,911 et l'erreur absolue moyenne de 17,8 nm, pour un déplacement moyen de 576 nm sur la base de test. Le témoin, qui répond toujours la moyenne, n'explique rien : son R² est de -0,037. La droite explique l'essentiel, mais la courbure lui échappe : un substitut utile doit faire mieux. C'est ce chiffre, mesuré sur des calculs que le modèle n'a jamais vus, que l'on a le droit d'annoncer.

## Pour aller plus loin : suivre la courbure, puis juger hors du plan de calcul

Le modèle de départ de l'étape 4 reste le score de référence du défi. Les cellules suivantes montrent comment le dépasser sans regarder le test pour choisir. La première compare, par validation croisée sur l'apprentissage seul, la droite, des polynômes de degrés 2 et 3, une droite sur l'inverse du ligament restant (`h - d0`, la largeur de matière de chaque côté du trou), un modèle des plus proches voisins et une forêt aléatoire. La deuxième est un test de domaine : on apprend sur les trois quarts de l'apprentissage les plus petits diamètres et on juge sur les diamètres plus grands, que le modèle n'a jamais rencontrés. La troisième mesure le modèle retenu une seule fois sur le test.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

In [ ]:
# Votre code :

Lecture du résultat : en validation croisée, la droite atteint 0,898, le polynôme de degré 2 0,998 et celui de degré 3 1,000 : le modèle retenu est « polynôme de degré 3 ». Mesuré une seule fois sur le test, il donne un R² de 1,000, soit 0,089 de plus que la droite. Les voisins et la forêt font aussi bien à l'intérieur du plan de calcul, mais le test de domaine les sépare : au-delà de 85 mm, que ces modèles n'ont jamais vus, la forêt tombe à un R² de -3,71 et les voisins à -3,98, parce qu'ils répètent la dernière valeur apprise ; le polynôme de degré 3 garde 0,96. Un substitut ne vaut que dans le domaine du plan de calcul : sortir de ce domaine demande un modèle dont la forme est celle du phénomène, ou de nouveaux calculs.

## Le score de référence

Le score publié est le R² du modèle de départ de l'étape 4 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = r2_score(y_test, modele.predict(X_test))
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE r2 = %.3f" % score)

## Synthèse

1. Un plan de calcul à une variable donne un jeu étroit : 100 calculs, une seule entrée utile. Les colonnes constantes, la colonne vide et les sorties du calcul (contrainte, durée) ne sont pas des entrées.
2. Une droite explique déjà 0,911 du déplacement, mais la courbure du phénomène lui échappe : le polynôme de degré 3 monte à 1,000 sur la base de test.
3. À l'intérieur du plan, plusieurs modèles se valent ; hors du plan, seule la forme compte : la forêt tombe à -3,71, le polynôme de degré 3 reste à 0,96.
4. Un calcul aberrant du solveur, dans les données, suffit à plafonner un score : on lit la forme d'une colonne avant de choisir la cible.